# Detector de piezas v2 — entrenamiento en Colab

Este notebook entrena **(a) el detector YOLO** de la arquitectura v2.

| | |
|---|---|
| Modelo base | `modelo_sintetico.pt` (YOLO11n de v1, **mismo orden de 12 clases**) |
| Paquete | `detector_v2.zip` (272.6 MB) en `Mi unidad/ajedrez_v2/` |
| Train | 3,802 tableros generados (1,544 sinteticos + 2,258 compuestos sobre fondo real) |
| Valid | 84 fotos reales: 12 recortes x 7 variantes |
| Test | 56 fotos reales: 8 recortes x 7 variantes |
| Epocas | 60 (con `patience=15`) |

**La regla que no se toca:** `valid` y `test` son SOLO fotos reales de la camara.
El train es todo lo generado. Si val o test tuvieran imagenes sinteticas, la
metrica mediria lo que el modelo ya vio y no serviria para decidir nada.

Antes de ejecutar: **Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU**.
Despues corre las celdas de arriba a abajo. Los pesos se guardan en tu Drive, asi
que si Colab se desconecta no pierdes el entrenamiento.

## 1. Verificar la GPU

In [ ]:
#@title 1. Verificar GPU y versiones
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU: ve a Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU")

## 2. Dependencias

Colab ya trae `torch`, `torchvision` y (casi siempre) `ultralytics`. **No se
reinstala nada por gusto**: instalar `ultralytics` a mano arrastra otra version
de `torch` y puede romper la GPU de la sesion. La celda solo instala si el
paquete no existe.

In [ ]:
#@title 2. Comprobar ultralytics sin tocar torch
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"],
        check=True,
    )

import ultralytics
print("ultralytics", ultralytics.__version__)
ultralytics.checks()

## 3. Traer el paquete de datos

Dos opciones:

- **Drive (recomendado):** sube `detector_v2.zip` a `Mi unidad/ajedrez_v2/` y
  deja `USAR_DRIVE = True`.
- **Subida directa:** pon `USAR_DRIVE = False` y elige el archivo cuando lo pida.
  Ojo: `files.upload()` de Colab no aguanta bien archivos de cientos de MB; para
  273 MB usa Drive.

El zip se descomprime en `/content` (disco local de la VM) y **no** en Drive:
leer miles de imagenes pequeñas montadas desde Drive es mucho mas lento.

In [ ]:
#@title 3. Descomprimir detector_v2.zip y contar por split
import shutil
import zipfile
from pathlib import Path

USAR_DRIVE = True
ZIP_EN_DRIVE = Path("/content/drive/MyDrive/ajedrez_v2/detector_v2.zip")
DESTINO = Path("/content/ajedrez_v2/detector")
ESPERADO = {"train": 3802, "val": 84, "test": 56}

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    origen = ZIP_EN_DRIVE
else:
    from google.colab import files
    subidos = files.upload()
    origen = Path(next(iter(subidos)))

if not origen.exists():
    raise FileNotFoundError(
        f"No encuentro {origen}. Sube detector_v2.zip a Mi unidad/ajedrez_v2/ "
        "o pon USAR_DRIVE = False."
    )

if DESTINO.exists():
    shutil.rmtree(DESTINO)
DESTINO.mkdir(parents=True)

with zipfile.ZipFile(origen) as zf:
    zf.extractall(DESTINO)

print("Contenido:", sorted(p.name for p in DESTINO.iterdir()))
for split, esperado in ESPERADO.items():
    n_img = len(list((DESTINO / "images" / split).glob("*.jpg")))
    n_lbl = len(list((DESTINO / "labels" / split).glob("*.txt")))
    estado = "OK" if n_img == esperado == n_lbl else "REVISAR"
    print(f"  {split:5s} imagenes={n_img:5d} etiquetas={n_lbl:5d} "
          f"(esperado {esperado:5d}) {estado}")

assert (DESTINO / "data.yaml").exists(), "Falta data.yaml en el zip"
assert (DESTINO / "weights" / "modelo_sintetico.pt").exists(), "Faltan los pesos base"

PROYECTO = Path("/content/drive/MyDrive/ajedrez_v2/detector")
PROYECTO.mkdir(parents=True, exist_ok=True)
print("Registro de entrenamientos:", PROYECTO)

In [ ]:
#@title 3b. Corregir `path` del data.yaml (REQUERIDO por Ultralytics)
# El zip trae `path` apuntando a la carpeta de descompresion de Colab. Si
# descomprimiste en otra ruta, esta celda lo reescribe con la correcta.
# Sin esto, Ultralytics resuelve `path` contra su datasets_dir GLOBAL
# (/content/datasets) y falla con: "missing path '/content/images/val'".
# Es el mismo error que costo una corrida entera en v1.
import re

RUTA_YAML = DESTINO / "data.yaml"
texto = RUTA_YAML.read_text(encoding="utf-8")
texto, cambios = re.subn(r"^path: .*$",
                         f"path: {DESTINO.as_posix()}", texto, flags=re.M)
RUTA_YAML.write_text(texto, encoding="utf-8")
print(f"path reescrito en {RUTA_YAML} ({cambios} cambio(s))")
print("\n".join(texto.splitlines()[:6]))
assert cambios >= 1, "data.yaml no tenia linea 'path:'"          

## 4. Auditar el dataset antes de entrenar

Esta celda es la que atrapa los errores silenciosos. Comprueba:

- imagen y etiqueta con el mismo nombre en los tres splits,
- lineas YOLO validas y clases dentro de rango,
- `data.yaml` con las **12 clases en el orden de `classes.txt`** (si el orden
  esta cruzado, el modelo entrena bien y llama "caballo" a una torre),
- que ninguna imagen este en dos splits a la vez.

In [ ]:
#@title 4. Auditoria del dataset
from collections import Counter
from pathlib import Path

import yaml

CLASES_ESPERADAS = [
    "peon_b", "torre_b", "caballo_b", "alfil_b", "dama_b", "rey_b",
    "peon_n", "torre_n", "caballo_n", "alfil_n", "dama_n", "rey_n",
]
RAIZ = DESTINO
cfg = yaml.safe_load((RAIZ / "data.yaml").read_text(encoding="utf-8"))
NOMBRES = [cfg["names"][i] for i in range(cfg["nc"])]
print("Suites:", cfg["train"], "|", cfg["val"], "|", cfg["test"])
print("Clases:", NOMBRES)
if NOMBRES != CLASES_ESPERADAS:
    raise SystemExit("data.yaml no respeta el orden de classes.txt: NO entrenes asi")

problemas = []
resumen = {}
vistos = {}

for split in ("train", "val", "test"):
    dir_img = RAIZ / "images" / split
    dir_lbl = RAIZ / "labels" / split
    imgs = {p.stem for p in dir_img.glob("*.jpg")}
    lbls = {p.stem for p in dir_lbl.glob("*.txt")}
    if imgs - lbls:
        problemas.append(f"{split}: {len(imgs - lbls)} imagenes sin etiqueta")
    if lbls - imgs:
        problemas.append(f"{split}: {len(lbls - imgs)} etiquetas sin imagen")

    conteo = Counter()
    vacias = 0
    invalidas = 0
    for txt in dir_lbl.glob("*.txt"):
        lineas = [l for l in txt.read_text(encoding="utf-8").splitlines() if l.strip()]
        if not lineas:
            vacias += 1
        for linea in lineas:
            partes = linea.split()
            if len(partes) != 5:
                invalidas += 1
                continue
            clase = int(partes[0])
            x, y, w, h = (float(v) for v in partes[1:])
            if not (0 <= clase < 12) or not all(0.0 < v <= 1.0 for v in (x, y, w, h)):
                invalidas += 1
                continue
            conteo[clase] += 1
    vistos[split] = imgs
    resumen[split] = {"imagenes": len(imgs), "cajas": sum(conteo.values()), "conteo": conteo}
    print(f"{split:5s} imagenes={len(imgs):5d} cajas={sum(conteo.values()):6d} "
          f"negativos={vacias:3d} invalidas={invalidas}")
    if invalidas:
        problemas.append(f"{split}: {invalidas} lineas invalidas")

for a in ("train", "val", "test"):
    for b in ("train", "val", "test"):
        if a < b and vistos[a] & vistos[b]:
            problemas.append(f"{len(vistos[a] & vistos[b])} imagenes repetidas entre {a} y {b}")

for split in ("val", "test"):
    faltan = [NOMBRES[i] for i in range(12) if resumen[split]["conteo"][i] == 0]
    if faltan:
        print(f"AVISO {split}: clases sin ejemplos -> {faltan}")

print()
if problemas:
    print("PROBLEMAS:")
    for p in problemas:
        print("  -", p)
    raise SystemExit("Corrige el dataset antes de entrenar.")
print("Dataset OK. Puedes entrenar.")

## 5. Configuracion del entrenamiento

`MODO = "fino"`: parte de `modelo_sintetico.pt` y ajusta todo el modelo con
`lr0 = 0.001`. El modelo base ya distingue las piezas; lo que falta es adaptarlo
a la camara y a las fichas nuevas.

Aumentos pensados para un tablero, no para sprites:

- `fliplr = 0.0`, `flipud = 0.0`: el tablero tiene orientacion **fija** y de ella
  depende el sentido del orden de clases (a1 abajo-izquierda). Voltear la imagen
  convierte una torre blanca en una negra de otra esquina. Las rotaciones de
  **ficha** ya las trae el dataset (el compositor gira cada pieza 0-360).
- `mosaic = 0.5` y `close_mosaic = 10`: mosaico suave y las ultimas 10 epocas con
  imagenes completas, para que el modelo vea el tablero entero antes de parar.
- `cache = "ram"`: 3,802 imagenes caben en la RAM de Colab y quita la lectura de
  disco del camino critico.
- `seed = 42`, `deterministic = True`: misma corrida, mismo resultado.

El proyecto de entrenamiento se escribe **en Drive** para no perder `best.pt` si
la sesion se cae.

In [ ]:
#@title 5. Configuracion
from pathlib import Path

MODO = "fino"          # "fino" (recomendado) | "dos_fases"
EPOCAS = 60
BATCH = 16             # YOLO11n a 640 px entra comodo en una T4
IMGSZ = 640
LR0 = 0.001
PATIENCE = 15
NOMBRE_RUN = "fino_v2"

BASE = DESTINO / "weights" / "modelo_sintetico.pt"
DATA = DESTINO / "data.yaml"
assert BASE.exists() and DATA.exists(), "Falta el paquete: vuelve a la celda 3"

print("Modo:", MODO)
print("Base:", BASE)
print("Epocas:", EPOCAS, "| batch:", BATCH, "| imgsz:", IMGSZ, "| lr0:", LR0)
print("Salida:", PROYECTO / NOMBRE_RUN)

In [ ]:
#@title 6. Entrenar
from pathlib import Path

from ultralytics import YOLO

COMUNES = dict(
    data=str(DATA),
    imgsz=IMGSZ,
    batch=BATCH,
    workers=2,
    cache="ram",
    seed=42,
    deterministic=True,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.5,
    close_mosaic=10,
    cos_lr=True,
    patience=PATIENCE,
    pretrained=True,
    plots=True,
    val=True,
    project=str(PROYECTO),
)

modelo = YOLO(str(BASE))

if MODO == "fino":
    resultado = modelo.train(
        epochs=EPOCAS,
        lr0=LR0,
        lrf=0.01,
        freeze=0,
        name=NOMBRE_RUN,
        exist_ok=True,
        save_period=5,
        **COMUNES,
    )
    PESO_FINAL = Path(resultado.save_dir) / "weights" / "best.pt"
else:
    fase1 = min(30, EPOCAS)
    modelo.train(epochs=fase1, lr0=0.01, freeze=10,
                 name=f"{NOMBRE_RUN}_fase1", exist_ok=True, save_period=5, **COMUNES)
    peso_fase1 = Path(modelo.trainer.save_dir) / "weights" / "best.pt"
    modelo = YOLO(str(peso_fase1))
    resultado = modelo.train(epochs=max(EPOCAS - fase1, 1), lr0=0.0005, lrf=0.01,
                             freeze=0, name=f"{NOMBRE_RUN}_fase2", exist_ok=True,
                             save_period=5, **COMUNES)
    PESO_FINAL = Path(resultado.save_dir) / "weights" / "best.pt"

print("Modelo final:", PESO_FINAL, "| existe:", PESO_FINAL.exists())

## 6. Evaluar sobre `test` (fotos reales, nunca vistas)

Este es el numero que vale. `valid` sirve para decidir cuando parar (lo usa
`patience`); `test` es la medida final.

**Linea base medida con el paquete de este notebook:** `modelo_sintetico.pt`
**sin ajustar** da `mAP50 = 0.040`, `precision = 0.088`, `recall = 0.128` sobre
las 56 imagenes reales de `test` (medido en CPU con `scripts/verificar_detector_v2.py`).
Es un modelo de otro dominio: encuentra poco y dispara cajas donde no hay ficha.
Si el fine-tuning no sube `mAP50` muy por encima de 0.55 y la precision por
encima de 0.60, algo esta mal: revisa las etiquetas antes de tocar
hiperparametros.

In [ ]:
#@title 7. Evaluar en test + metricas por clase
from ultralytics import YOLO

modelo_eval = YOLO(str(PESO_FINAL))
metricas = modelo_eval.val(data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH,
                           plots=True, project=str(PROYECTO), name="eval_test",
                           exist_ok=True)

print()
print(f"TEST  mAP50   : {metricas.box.map50:.4f}")
print(f"TEST  mAP50-95: {metricas.box.map:.4f}")
print(f"TEST  precision: {metricas.box.mp:.4f}")
print(f"TEST  recall   : {metricas.box.mr:.4f}")

print()
print(f"{'clase':12s} {'P':>7s} {'R':>7s} {'mAP50-95':>9s}")
for i, nombre in metricas.names.items():
    p = metricas.box.p[i] if i < len(metricas.box.p) else 0.0
    r = metricas.box.r[i] if i < len(metricas.box.r) else 0.0
    m = metricas.box.maps[i] if i < len(metricas.box.maps) else 0.0
    print(f"{nombre:12s} {p:7.3f} {r:7.3f} {m:9.3f}")

## 7. Confusiones reales -> predichas

La matriz de confusion dice que clases se confunden entre si (por ejemplo
`torre_b -> dama_b`). Es lo que la metrica global no te cuenta.

In [ ]:
#@title 8. Confusiones mas frecuentes
import numpy as np

cm = getattr(getattr(metricas, "confusion_matrix", None), "matrix", None)
if cm is None:
    print("Esta version de Ultralytics no expuso la matriz de confusion.")
else:
    cm = np.asarray(cm)
    nombres = metricas.names
    pares = []
    for real in range(min(len(nombres), cm.shape[0])):
        for pred in range(min(len(nombres), cm.shape[1])):
            if real != pred and cm[real, pred] > 0:
                pares.append((float(cm[real, pred]), nombres[real], nombres[pred]))
    pares.sort(reverse=True)
    print("Confusiones mas frecuentes (real -> predicha):")
    if not pares:
        print("  Ninguna. Excelente.")
    for cantidad, real, pred in pares[:12]:
        print(f"  {real:12s} -> {pred:12s} {cantidad:g}")

## 8. Guardar todo en Drive

Sin esta celda, si Colab cierra la sesion se pierde el entrenamiento. `last.pt`
permite reanudar; `best.pt` es el modelo que se usa despues.

In [ ]:
#@title 9. Copiar modelo y curvas a Drive
import shutil
from pathlib import Path

SALIDA = PROYECTO / "entrega"
SALIDA.mkdir(parents=True, exist_ok=True)

shutil.copy2(PESO_FINAL, SALIDA / "best.pt")

ultimo = PESO_FINAL.parent / "last.pt"
if ultimo.exists():
    shutil.copy2(ultimo, SALIDA / "last.pt")

run_dir = PESO_FINAL.parent.parent
for patron in ("results.csv", "confusion_matrix.png", "confusion_matrix_normalized.png",
               "PR_curve.png", "F1_curve.png", "results.png"):
    for archivo in run_dir.glob(patron):
        shutil.copy2(archivo, SALIDA / archivo.name)

shutil.copy2(DATA, SALIDA / "data.yaml")

print("Guardado en", SALIDA)
for p in sorted(SALIDA.iterdir()):
    print(f"  {p.name:38s} {p.stat().st_size / 1024:8.1f} KB")

## 9. Prueba visual rapida

Dibuja las predicciones sobre una foto real de `test`. Es lo que la matriz de
confusion no te cuenta: cajas dobles, fichas partidas, sombras detectadas.

In [ ]:
#@title 10. Predecir sobre una foto de test
import random

from IPython.display import display
from PIL import Image

fotos = sorted((DESTINO / "images" / "test").glob("*.jpg"))
elegida = random.choice(fotos)
print("Foto:", elegida.name)

prediccion = modelo_eval.predict(str(elegida), imgsz=IMGSZ, conf=0.40, verbose=False)[0]
salida = Path("/content/prediccion_v2.jpg")
Image.fromarray(prediccion.plot()[..., ::-1]).save(salida)
display(Image.open(salida))
print(f"{len(prediccion.boxes)} detecciones")

## Siguiente paso (no es este notebook)

Con `best.pt` ya entrenado se etiquetan automaticamente las **66 fotos reales de
medio juego** que hoy no tienen etiqueta (`dataset/reales/recortes/800x800/02_medio_juego`).
El plan esta en `colab/README.md`, seccion **"Siguiente paso"**: predecir con este
modelo, revisar a ojo las cajas dudosas, y volver a entrenar con ellas. Ese es el
camino para que `test` deje de ser solo posicion inicial y vacio.

Para comprobar el modelo en local (CPU, sin GPU) antes de usarlo:

```bash
python scripts/verificar_detector_v2.py --bundle dataset/colab_bundle/detector_v2.zip
```

## Si algo falla

| Sintoma | Causa | Solucion |
|---|---|---|
| `SIN GPU` | Entorno sin GPU | Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU |
| `No encuentro ...detector_v2.zip` | Ruta de Drive distinta | Revisa `ZIP_EN_DRIVE` o pon `USAR_DRIVE = False` |
| `data.yaml no respeta el orden` | Zip editado a mano | Vuelve a descargar el `detector_v2.zip` del proyecto |
| `CUDA out of memory` | `BATCH` alto | Baja `BATCH` a 8 |
| mAP50 alto en valid y bajo en test | Sobreajuste a val | Baja `EPOCAS`, sube `PATIENCE`, revisa etiquetas de test |
| Se perdio el entrenamiento | Colab cerro la sesion | Reentrena con `resume` desde `PROYECTO/fino_v2/weights/last.pt` |